In [ ]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from delta.tables import DeltaTable

print("[INIT] Silver notebook started.")
print("[INIT] Imports completed successfully.")

In [ ]:
# ============================================================
# CELL 2 — PIPELINE PARAMETERS
# ============================================================
#
# Pipeline should pass:
#   load_type → Full
#   run_id    → @pipeline().RunId
#
# source_table/target_table are no longer pipeline parameters —
# this notebook processes ALL Bronze sources internally, once
# per run, since each source has a different nested JSON shape
# under the hood but the same flattening pattern applies.
# ============================================================

load_type = "Full"
run_id = "MANUAL_TEST"

print("[PARAMETERS] Silver notebook parameters initialized.")
print(f"[PARAMETERS] Load Type : {load_type}")
print(f"[PARAMETERS] Run ID    : {run_id}")

In [ ]:
# ============================================================
# CELL 3 — PARAMETER VALIDATION
# ============================================================

required_parameters = {
    "load_type": load_type,
    "run_id": run_id
}

for parameter_name, parameter_value in required_parameters.items():

    if parameter_value is None or str(parameter_value).strip() == "":
        raise ValueError(
            f"Required parameter '{parameter_name}' is missing."
        )

valid_load_types = ["Full", "Incremental"]

if load_type not in valid_load_types:
    raise ValueError(
        f"Invalid load_type '{load_type}'. Expected one of {valid_load_types}."
    )

print("[VALIDATION] All parameters are valid.")

In [ ]:

# ============================================================
# CELL 4 — SOURCE CONFIGURATION
# ============================================================
#
# Maps each Bronze table to its logical source_type tag.
# Add new rows here when new Bronze sources come online
# (e.g. scorecard, once its two-stage fetch is built).
# ============================================================

bronze_sources = [
    {"bronze_table": "bronze_matches",  "source_type": "recent"},
    {"bronze_table": "bronze_upcoming", "source_type": "upcoming"},
]

silver_table = "silver_matches"

print(f"[CONFIG] Bronze sources to process: {len(bronze_sources)}")
for src in bronze_sources:
    print(f"[CONFIG]   {src['bronze_table']} -> source_type={src['source_type']}")

In [ ]:

# ============================================================
# CELL 5 — HELPER: READ LATEST BRONZE SNAPSHOT
# ============================================================
#
# Bronze keeps one row per pipeline run (full JSON snapshot).
# Silver only cares about the MOST RECENT snapshot per table.
# ============================================================

def get_latest_bronze_row(table_name):

    df = spark.table(table_name)
    latest_ts = df.agg(F.max("ingested_at")).collect()[0][0]

    if latest_ts is None:
        raise ValueError(f"Bronze table '{table_name}' has no rows.")

    return df.filter(F.col("ingested_at") == latest_ts)

In [ ]:


# ============================================================
# CELL 6 — FLATTEN EACH BRONZE SOURCE
# ============================================================

flattened_frames = []

for src in bronze_sources:

    bronze_table = src["bronze_table"]
    source_type = src["source_type"]

    print(f"[SILVER] Processing {bronze_table} ({source_type})...")

    try:
        bronze_row_df = get_latest_bronze_row(bronze_table)
    except ValueError as e:
        print(f"[SILVER] Skipping {bronze_table}: {str(e)}")
        continue

    row = bronze_row_df.select("response_json", "ingested_at", "run_id").collect()[0]
    json_str = row["response_json"]
    bronze_ingested_at = row["ingested_at"]
    bronze_run_id = row["run_id"]

    parsed_df = spark.read.json(spark.sparkContext.parallelize([json_str]))

    if "typeMatches" not in parsed_df.columns:
        print(f"[SILVER] WARNING: 'typeMatches' not found in {bronze_table}. Skipping.")
        continue

    exploded = (
        parsed_df
        .select(F.explode("typeMatches").alias("type_match"))
        .select(
            F.col("type_match.matchType").alias("match_type"),
            F.explode("type_match.seriesMatches").alias("series_match")
        )
        .select(
            "match_type",
            F.col("series_match.seriesAdWrapper.seriesId").alias("series_id"),
            F.col("series_match.seriesAdWrapper.seriesName").alias("series_name"),
            F.explode("series_match.seriesAdWrapper.matches").alias("match")
        )
    )

    # matchScore lives at match.matchScore, NOT match.matchInfo.matchScore
    match_fields = [f.name for f in exploded.schema["match"].dataType.fields]
    has_score = "matchScore" in match_fields

    if has_score:
        score_cols = [
            F.col("match.matchScore.team1Score.inngs1.runs").alias("team1_runs"),
            F.col("match.matchScore.team1Score.inngs1.wickets").alias("team1_wickets"),
            F.col("match.matchScore.team1Score.inngs1.overs").alias("team1_overs"),
            F.col("match.matchScore.team2Score.inngs1.runs").alias("team2_runs"),
            F.col("match.matchScore.team2Score.inngs1.wickets").alias("team2_wickets"),
            F.col("match.matchScore.team2Score.inngs1.overs").alias("team2_overs"),
        ]
    else:
        score_cols = [
            F.lit(None).cast("long").alias("team1_runs"),
            F.lit(None).cast("long").alias("team1_wickets"),
            F.lit(None).cast("string").alias("team1_overs"),
            F.lit(None).cast("long").alias("team2_runs"),
            F.lit(None).cast("long").alias("team2_wickets"),
            F.lit(None).cast("string").alias("team2_overs"),
        ]

    flat = exploded.select(
        "match_type", "series_id", "series_name",
        F.col("match.matchInfo.matchId").cast("long").alias("match_id"),
        F.trim(F.col("match.matchInfo.matchDesc")).alias("match_desc"),
        F.col("match.matchInfo.matchFormat").alias("match_format"),
        F.col("match.matchInfo.status").alias("status"),
        F.col("match.matchInfo.state").alias("state"),
        F.col("match.matchInfo.startDate").cast("long").alias("start_date_epoch"),
        F.col("match.matchInfo.endDate").cast("long").alias("end_date_epoch"),
        F.col("match.matchInfo.team1.teamId").alias("team1_id"),
        F.trim(F.col("match.matchInfo.team1.teamName")).alias("team1_name"),
        F.col("match.matchInfo.team2.teamId").alias("team2_id"),
        F.trim(F.col("match.matchInfo.team2.teamName")).alias("team2_name"),
                F.col("match.matchInfo.team1.teamSName").alias("team1_sname"),
        F.col("match.matchInfo.team2.teamSName").alias("team2_sname"),
        F.trim(F.col("match.matchInfo.venueInfo.ground")).alias("venue_ground"),
        F.trim(F.col("match.matchInfo.venueInfo.city")).alias("venue_city"),
        *score_cols
    ).withColumn(
        "source_type", F.lit(source_type)
    ).withColumn(
        "bronze_run_id", F.lit(bronze_run_id)
    ).withColumn(
        "bronze_ingested_at", F.lit(bronze_ingested_at)
    ).withColumn(
        "silver_run_id", F.lit(run_id)
    ).withColumn(
        "silver_updated_at", F.current_timestamp()
    )

    row_count = flat.count()
    print(f"[SILVER]   {bronze_table}: {row_count} matches flattened.")

    flattened_frames.append(flat)

if len(flattened_frames) == 0:
    raise RuntimeError("No Bronze sources produced any flattened rows. Aborting Silver write.")

print(f"[SILVER] Successfully flattened {len(flattened_frames)} Bronze source(s).")

In [ ]:
# ============================================================
# CELL 7 — COMBINE ALL SOURCES
# ============================================================

silver_df = flattened_frames[0]
for f in flattened_frames[1:]:
    silver_df = silver_df.unionByName(f, allowMissingColumns=True)

# Convert epoch millis -> real timestamps
silver_df = silver_df \
    .withColumn("start_date", (F.col("start_date_epoch") / 1000).cast("timestamp")) \
    .withColumn("end_date",   (F.col("end_date_epoch") / 1000).cast("timestamp")) \
    .drop("start_date_epoch", "end_date_epoch")

# Null-handling for string columns (Silver-layer standardization)
for field in silver_df.schema.fields:
    if isinstance(field.dataType, StringType):
        silver_df = silver_df.withColumn(
            field.name, F.coalesce(F.col(field.name), F.lit(""))
        )

# Drop rows with no match_id — can't upsert without a key
before_null_check = silver_df.count()
silver_df = silver_df.filter(F.col("match_id").isNotNull())
after_null_check = silver_df.count()

if before_null_check != after_null_check:
    print(f"[QUALITY] Dropped {before_null_check - after_null_check} rows with null match_id.")

total_count = silver_df.count()
print(f"[SILVER] Total combined rows to upsert: {total_count}")
silver_df.show(5, truncate=False)

In [ ]:
# # ============================================================
# # CELL 8 — UPSERT INTO silver_matches (merge on match_id)
# # ============================================================

# print(f"[SILVER] Writing to {silver_table}...")

# try:

#     if not spark.catalog.tableExists(silver_table):

#         print(f"[SILVER] {silver_table} does not exist. Creating...")

#         silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

#     else:

#         print(f"[SILVER] Merging into existing {silver_table}...")

#         target = DeltaTable.forName(spark, silver_table)

#         target.alias("t").merge(
#             silver_df.alias("s"),
#             "t.match_id = s.match_id"
#         ).whenMatchedUpdateAll() \
#          .whenNotMatchedInsertAll() \
#          .execute()

# except Exception as e:

#     raise RuntimeError(f"Failed to write Silver table '{silver_table}'. Error: {str(e)}")
# ============================================================
# CELL 8 — UPSERT INTO silver_matches (merge on match_id)
# ============================================================

print(f"[SILVER] Writing to {silver_table}...")

try:

    if not spark.catalog.tableExists(silver_table):

        print(f"[SILVER] {silver_table} does not exist. Creating...")

        silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

    else:

        print(f"[SILVER] Merging into existing {silver_table}...")

        target = DeltaTable.forName(spark, silver_table)

        spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

        target.alias("t").merge(
            silver_df.alias("s"),
            "t.match_id = s.match_id"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()

except Exception as e:

    raise RuntimeError(f"Failed to write Silver table '{silver_table}'. Error: {str(e)}")

In [ ]:
# ============================================================
# CELL 9 — FINAL VALIDATION AND LOGGING
# ============================================================

silver_final_count = spark.table(silver_table).count()

print("=" * 60)
print("SILVER PIPELINE COMPLETED")
print("=" * 60)
print(f"Sources Processed  : {[s['bronze_table'] for s in bronze_sources]}")
print(f"Rows Upserted       : {total_count}")
print(f"Silver Table Count  : {silver_final_count}")
print(f"Pipeline Run ID     : {run_id}")
print(f"Status              : SUCCESS")
print("=" * 60)

In [ ]:
# # ============================================================
# # CELL 10 — FINAL VALIDATION AND COMPLETION
# # ============================================================

# print("[FINAL] Validating Silver table...")

# final_df = spark.table(target_table)

# final_count = final_df.count()

# print(f"[FINAL] Target table : {target_table}")
# print(f"[FINAL] Final count  : {final_count}")
# print(f"[FINAL] Run ID       : {run_id}")

# if final_count == 0:
#     raise ValueError(
#         f"Silver table '{target_table}' contains 0 records "
#         "after the write operation."
#     )

# print("[FINAL] Silver pipeline completed successfully.")